<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">K-Means Clustering</h1></center>

K-Means partitions data into k clusters by minimising **within-cluster sum of squares**.

### Topics:
1. [Theory — Lloyd's Algorithm](#1)
2. [Dataset & Visualization](#2)
3. [Choosing k — Elbow & Silhouette](#3)
4. [K-Means Model](#4)
5. [Cluster Profiling](#5)
6. [Evaluation](#6)

In [ ]:
import seaborn as sns
km_colors = ['#ffbe0b', '#fb5607', '#ff006e', '#8338ec', '#3a86ff']
print("K-Means Colors:")
sns.palplot(sns.color_palette(km_colors))

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.datasets import make_blobs, load_iris
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline
np.random.seed(42)
print("Libraries loaded.")

## Let's Start!

<a id = "1"></a>
<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Theory — Lloyd's Algorithm</h1></center>

## K-Means Objective

Minimise **Within-Cluster Sum of Squares (WCSS)** also called **inertia**:

$$\min_{C_1,\ldots,C_k} \sum_{i=1}^{k} \sum_{x \in C_i} \|x - \mu_i\|^2$$

## Lloyd's Algorithm

1. **Initialise** k centroids (randomly or via k-means++)
2. **Assign** each point to the nearest centroid
3. **Update** centroids to the mean of their assigned points
4. **Repeat** steps 2-3 until centroids stop moving

**k-means++ initialisation** spreads initial centroids apart — avoids bad local minima.

<SCRIPT SRC='https://cdn.mathjax.org/mathjax/latest/MathJax.js?config=TeX-AMS-MML_HTMLorMML'></SCRIPT>
<SCRIPT>MathJax.Hub.Config({ tex2jax: {inlineMath: [['$','$'],['\\(','\\)']]}});</SCRIPT>

<a id = "2"></a>
<center><h1 style = "background:#ffbe0b ;color:black;border:0;font-weight:bold">Information About Dataset</h1></center>

**Iris** — 150 samples, 4 features (sepal/petal length/width), 3 species.

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame
print(f"Shape: {df.shape}")
print(f"Features: {iris.feature_names}")
print(f"Classes: {iris.target_names}")
df.describe()

<center><h1 style = "background:#fb5607 ;color:white;border:0;font-weight:bold">Data Visualization</h1></center>

In [ ]:
plt.figure(dpi=100)
sns.pairplot(df, hue="target",
             palette={0: "#ffbe0b", 1: "#ff006e", 2: "#3a86ff"},
             diag_kind="kde")
plt.suptitle("Iris Pairplot", y=1.01)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5), dpi=100)
sns.heatmap(df.drop(columns="target").corr(numeric_only=True),
            annot=True, fmt=".2f", cmap="YlOrRd", linewidths=0.5)
plt.title("Feature Correlation Heatmap")
plt.show()

<center><h1 style = "background:#ff006e ;color:white;border:0;font-weight:bold">Train-Test Split (Unsupervised — No Labels Used)</h1></center>

In [ ]:
X_iris = df.drop(columns="target")
y_iris = df["target"]  # only for evaluation, not training

sc = StandardScaler()
X_scaled = sc.fit_transform(X_iris)
print(f"Features shape: {X_scaled.shape}")
print("Note: K-Means is unsupervised — labels are only used for post-hoc evaluation.")

<a id = "3"></a>
<center><h1 style = "background:#8338ec ;color:white;border:0;font-weight:bold">Choosing k — Elbow & Silhouette</h1></center>

In [ ]:
K_range = range(2, 11)
inertias, sil_scores = [], []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

best_k = list(K_range)[np.argmax(sil_scores)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=100)

axes[0].plot(list(K_range), inertias, "o-", color="#ffbe0b")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia (WCSS)")
axes[0].set_title("Elbow Method", fontweight="bold")

axes[1].plot(list(K_range), sil_scores, "s-", color="#ff006e")
axes[1].axvline(best_k, color="gray", ls="--", label=f"Best k={best_k}")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score vs k", fontweight="bold")
axes[1].legend()

plt.tight_layout(); plt.show()
print(f"Best k by silhouette: {best_k}")

<a id = "4"></a>
<center><h1 style = "background:#3a86ff ;color:white;border:0;font-weight:bold">K-Means Model</h1></center>

In [ ]:
best_km = KMeans(n_clusters=best_k, n_init=20, random_state=42)
best_km.fit(X_scaled)
df["cluster"] = best_km.labels_

print(f"Inertia : {best_km.inertia_:.4f}")
print(f"Silhouette: {silhouette_score(X_scaled, best_km.labels_):.4f}")
print(f"\nCluster sizes:\n{pd.Series(best_km.labels_).value_counts().sort_index()}")

In [ ]:
# Scatter of first 2 features coloured by cluster
plt.figure(figsize=(8, 5), dpi=100)
palette = ["#ffbe0b", "#ff006e", "#3a86ff", "#8338ec"]
for c in range(best_k):
    mask = df["cluster"] == c
    plt.scatter(X_scaled[mask, 0], X_scaled[mask, 1],
                color=palette[c], label=f"Cluster {c}", s=40, alpha=0.8)
centers = best_km.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c="black", marker="X", s=150, zorder=5,
            label="Centroids")
plt.xlabel("Feature 1 (scaled)"); plt.ylabel("Feature 2 (scaled)")
plt.title(f"K-Means Clusters (k={best_k})", fontweight="bold")
plt.legend(); plt.show()

<a id = "5"></a>
<center><h1 style = "background:#fb5607 ;color:white;border:0;font-weight:bold">Cluster Profiling</h1></center>

In [ ]:
profile = df.groupby("cluster").mean(numeric_only=True).round(2)
print("Cluster Profiles (feature means):")
print(profile.to_string())

# Heatmap
fig, ax = plt.subplots(figsize=(8, 3), dpi=100)
im = ax.imshow(profile.drop(columns="target", errors="ignore").values,
               aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(iris.feature_names)))
ax.set_xticklabels(iris.feature_names, rotation=30, ha="right")
ax.set_yticks(range(best_k))
ax.set_yticklabels([f"Cluster {i}" for i in range(best_k)])
plt.colorbar(im, ax=ax)
ax.set_title("Cluster Profile Heatmap", fontweight="bold")
for i in range(best_k):
    for j, val in enumerate(profile.drop(columns="target", errors="ignore").values[i]):
        ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=9)
plt.tight_layout(); plt.show()

<a id = "6"></a>
<center><h1 style = "background:#8338ec ;color:white;border:0;font-weight:bold">Evaluation of Model</h1></center>

In [ ]:
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(y_iris, best_km.labels_)
sil = silhouette_score(X_scaled, best_km.labels_)

results = [best_km.inertia_, sil, ari]
pd.DataFrame({"Metric": ["Inertia (WCSS)", "Silhouette Score", "Adjusted Rand Index (vs true labels)"],
              "Score": results}).round(4)

In [ ]:
cross = pd.crosstab(y_iris.map({0:"setosa",1:"versicolor",2:"virginica"}),
                    df["cluster"].map({i: f"Cluster {i}" for i in range(best_k)}))
print("Cluster vs True Species:")
print(cross)